# Retrieval reliability on HotpotQA (distractor) - UNTESTED TEMPLATE
I could not run this in my build environment (no HuggingFace access). Needs Internet On. Measures supporting-fact recall@k for BM25, dense (bge-small) and RRF, and whether cross-path agreement predicts retrieval success (the signal the self-audit method relies on).

In [ ]:
!pip -q install rank_bm25 sentence-transformers datasets scikit-learn

In [ ]:
import numpy as np, re
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
ds = load_dataset("hotpot_qa", "distractor", split="validation").select(range(1500))
enc = SentenceTransformer("BAAI/bge-small-en-v1.5")
K = 5
def rrf(ls, k=K, c=60):
    s = {}
    for L in ls:
        for r, i in enumerate(L): s[i] = s.get(i, 0) + 1 / (c + r + 1)
    return [i for i, _ in sorted(s.items(), key=lambda x: -x[1])[:k]]
rows = []
for ex in ds:
    titles = ex["context"]["title"]; paras = [" ".join(s) for s in ex["context"]["sentences"]]
    gold = set(ex["supporting_facts"]["title"])
    bm = BM25Okapi([re.findall(r"\w+", p.lower()) for p in paras]); sc = bm.get_scores(re.findall(r"\w+", ex["question"].lower()))
    L1 = list(np.argsort(-sc)[:K])
    E = enc.encode(paras, normalize_embeddings=True); q = enc.encode([ex["question"]], normalize_embeddings=True)[0]
    L2 = list(np.argsort(-(E @ q))[:K]); F = rrf([L1, L2])
    rec = lambda L: len(gold & {titles[i] for i in L}) / len(gold)
    rows.append((rec(L1), rec(L2), rec(F), len(set(L1) & set(L2)) / K, float(np.sort(sc)[-1] - np.sort(sc)[-2]), float(np.sort(E @ q)[-1] - np.sort(E @ q)[-2]), rec(F) == 1.0))
A = np.array(rows, float); print("supporting-fact recall@5  bm25 %.3f  dense %.3f  rrf %.3f" % tuple(A[:, :3].mean(0)))
X, y = A[:, 3:6], A[:, 6]
p = cross_val_predict(LogisticRegression(max_iter=500), X, y, cv=5, method="predict_proba")[:, 1]
print("AUC of agreement features predicting 'all gold paragraphs retrieved':", round(roc_auc_score(y, p), 3))